# Research Prototype: AUTOSAR Document Chunking

This notebook demonstrates the **research / exploratory** stage of developing
the chunking component. It shows the messy, iterative, print-debug style
typical of a Jupyter research environment.

**Part 1** is the raw research prototype — no classes, printed debug output,
global variables, quick-and-dirty regex.

**Part 2** shows how the same logic was refactored into the production
module `app/services/ingestion/chunker.py` — typed dataclasses, structured
logging, configuration injection, unit tests.

---
## Part 1 — Research Prototype (messy, exploratory)

In [ ]:
# Research prototype: quick chunking experiment
# No classes, no error handling, print everything

import re

CHUNK_SIZE = 100   # tokens (rough)
OVERLAP = 10

sample_text = """
7 API Specification
The Com module provides signal-based communication services.
It handles routing of signals between application and communication stack.
[SWS_Com_00001] The Com module shall provide signal routing.
[SWS_Com_00002] The Com module shall support I-PDU grouping.

7.1 Com_Init
This function initializes the Com module. It must be called once at startup.
[SWS_Com_00432] Com_Init shall initialize all signal buffers.

7.2 Com_DeInit
Stops all I-PDU transmissions and resets internal state.
[SWS_Com_00500] Com_DeInit shall stop all running timers.
"""

# Step 1: find section headings — research version: just split on numbered lines
heading_re = re.compile(r'^(\d+(?:\.\d+)*)\s+(.+)$', re.MULTILINE)
sws_re = re.compile(r'\[SWS_[A-Za-z]+_\d+\]')

headings = heading_re.findall(sample_text)
print('Found headings:', headings)

In [ ]:
# Step 2: split text by headings — no overlap logic yet
sections = []
matches = list(heading_re.finditer(sample_text))
for i, m in enumerate(matches):
    start = m.start()
    end = matches[i+1].start() if i + 1 < len(matches) else len(sample_text)
    section_text = sample_text[start:end].strip()
    heading = m.group(0).strip()
    sws_ids = sws_re.findall(section_text)
    print(f'--- Section: {heading} ---')
    print(f'  chars={len(section_text)}, SWS IDs={sws_ids}')
    sections.append({'heading': heading, 'text': section_text, 'sws': sws_ids})

print(f'\nTotal sections: {len(sections)}')

In [ ]:
# Step 3: naive token estimate (whitespace split) and chunking — research style
def estimate_tokens(text):
    return max(1, len(text.split()))

chunks = []
for sec in sections:
    words = sec['text'].split()
    i = 0
    while i < len(words):
        chunk_words = words[i:i + CHUNK_SIZE]
        chunk_text = ' '.join(chunk_words)
        chunks.append({'text': chunk_text, 'section': sec['heading'],
                        'tokens': len(chunk_words)})
        i += CHUNK_SIZE - OVERLAP   # basic overlap

print(f'Total chunks: {len(chunks)}')
for c in chunks:
    print(f"  [{c['section']}] tokens={c['tokens']}")

In [ ]:
# Step 4: observe problem — SWS IDs get split across chunks!
for c in chunks:
    found = sws_re.findall(c['text'])
    if found:
        print(f"SWS IDs in chunk '{c['section']}': {found}")
    else:
        print(f"No SWS IDs in chunk '{c['section']}' — may have been split!")

### Observations from prototype

1. The basic whitespace-token chunking works but **SWS IDs can be split** across
   chunk boundaries when a requirement ID happens to fall exactly at the chunk edge.
2. No metadata (page number, document name, chunk index) is stored — makes
   citation impossible.
3. Token estimation is crude (whitespace split ≠ real tokenizer).
4. No logging — failures are silent.
5. Global variables make testing and reuse impossible.

These problems were fixed in the production code (Part 2).

---
## Part 2 — Production Code Contrast

The same chunking logic now lives in `app/services/ingestion/chunker.py`.
Here we call it directly to show the clean, testable, production interface.

In [ ]:
# Make sure the project root is on the path
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Production imports — typed dataclasses, config-driven, structured logging
from app.services.ingestion.parser import ParsedDocument, ParsedPage
from app.services.ingestion.chunker import chunk_document, Chunk
from app.monitoring.logging_config import setup_logging

setup_logging('WARNING')   # suppress INFO noise in notebook output

In [ ]:
# Build a minimal ParsedDocument (same content as research prototype)
page = ParsedPage(
    page_number=1,
    text=sample_text,
    headings=['7 API Specification', '7.1 Com_Init', '7.2 Com_DeInit'],
    requirement_ids=['[SWS_Com_00001]', '[SWS_Com_00002]',
                     '[SWS_Com_00432]', '[SWS_Com_00500]'],
    has_tables=False,
)

doc = ParsedDocument(
    document_name='autosar_com_spec.pdf',
    file_path='/tmp/autosar_com_spec.pdf',
    total_pages=1,
    pages=[page],
)

# chunk_document is a pure function — easy to unit test, config-injected
chunks: list[Chunk] = chunk_document(doc, chunk_size=100, chunk_overlap=10)
print(f'Production chunker produced {len(chunks)} chunks')

In [ ]:
# Every chunk carries rich metadata — citations are now possible
for c in chunks:
    print(
        f'  chunk_id={c.chunk_id}'
        f'  section={c.section!r}'
        f'  tokens={c.token_count}'
        f'  sws_ids={c.requirement_ids}'
        f'  page={c.page_number}-{c.page_end}'
    )

In [ ]:
# Metadata is ChromaDB-ready — no None values, flat types only
meta = chunks[0].to_metadata_dict()
print('Chunk 0 metadata dict:')
for k, v in meta.items():
    print(f'  {k}: {v!r}  ({type(v).__name__})')

### Key differences: research vs production

| Aspect | Research (Part 1) | Production (`chunker.py`) |
|--------|-------------------|---------------------------|
| Structure | Global variables, no classes | `Chunk` dataclass, pure functions |
| Token counting | `len(text.split())` | `tiktoken` BPE tokenizer |
| Metadata | None | page, section, SWS IDs, chunk index |
| Error handling | Silent failures | `logger.error` with context |
| Logging | `print()` | `structlog` JSON with correlation ID |
| Configuration | Hard-coded globals | Injected via `Settings` / `get_settings()` |
| Testability | Manual runs only | 15 unit tests in `tests/test_chunker.py` |
| SWS ID preservation | May split across chunks | Extracted and stored per chunk |
| ChromaDB compatible | No | Yes — `to_metadata_dict()` produces flat types |